# **01 - Data Preparation**

This notebook covers the end-to-end preparation for the TFG:
1. **Data Loading**: Integration of XLSX metadata and CSV text files.
2. **Multilabel Reconstruction**: Mapping SDGs to each announcement.
3. **Filtering**: Keeping only announcements with at least one SDG.
4. **Dual Preprocessing**: Specific cleaning paths for Machine Learning (ML) and Deep Learning (DL).
5. **Stratified Splitting & Export**: Saving consistent train/val/test sets for comparison.

In [1]:
import sys
import pandas as pd
import os
import re
import unicodedata
import spacy
import pickle
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

sys.path.append('../src')
from utils import RAW_METADATA_PATH, RAW_TEXT_CSV_DIR, OUTPUT_DIR,PROCESSED_ML_DIR, PROCESSED_DL_DIR, RANDOM_SEED, TEST_SIZE, VAL_SIZE
from schema import Metadata, Description, ProcessedData

os.makedirs(PROCESSED_ML_DIR, exist_ok=True)
os.makedirs(PROCESSED_DL_DIR, exist_ok=True)

nlp = spacy.load("ca_core_news_lg")

## **1. Data Integration**

In [2]:
# 1.1 Load Metadata
df_meta = pd.read_excel(RAW_METADATA_PATH, dtype=str)
df_meta[Metadata.ODS] = df_meta[Metadata.ODS].str[:6].str.strip()

df_grouped_ods = df_meta.groupby(Metadata.ID)[Metadata.ODS].apply(
    lambda x: list(set(v for v in x if pd.notna(v)))
).reset_index()
df_grouped_ods.rename(columns={Metadata.ODS: ProcessedData.ODS_LIST}, inplace=True)
df_meta_unique = df_meta.drop(columns=[Metadata.ODS]).drop_duplicates(subset=[Metadata.ID])
df_meta = df_meta_unique.merge(df_grouped_ods, on=Metadata.ID, how='left')

# 1.2 Load Texts from multiple CSVs
csv_files = [os.path.join(RAW_TEXT_CSV_DIR, f) for f in os.listdir(RAW_TEXT_CSV_DIR) if f.endswith('.csv')]
df_texts = pd.concat([pd.read_csv(f, dtype=str) for f in csv_files], ignore_index=True)

# 1.3 Merge Metadata and Texts on ID
df = df_meta.merge(df_texts[[Description.ID_REGISTRE, Description.TEXT]].rename(columns={Description.ID_REGISTRE: Metadata.ID_REGISTRE}), on=Metadata.ID_REGISTRE, how='left')

# Create full_text field (Title + Body)
df[ProcessedData.FULL_TEXT] = df[Metadata.TITLE].fillna('') + " " + df[Description.TEXT].fillna('')

# Export df for EDA
df.to_parquet(os.path.join(OUTPUT_DIR, 'full_dataset.parquet'), index=False)

print(f"Initial dataset size: {len(df)}")

Initial dataset size: 35520


## **2. Filtering and Differentiated Processing**

In [3]:
# Filter: Keep only announcements with at least one ODS
df_filtered = df[df[ProcessedData.ODS_LIST].apply(len) > 0].copy()
print(f"Filtered dataset size (announcements with ODSs): {len(df_filtered)}")

Filtered dataset size (announcements with ODSs): 19284


In [4]:
def clean_base(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[\x00-\x1f\x7f]', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    return text

def process_ml(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^a-zàáèéíïòóúüçñ·\s]', ' ', text)
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and len(token.text) > 2]
    return " ".join(tokens)

In [5]:
df_filtered[ProcessedData.FULL_TEXT] = df_filtered[ProcessedData.FULL_TEXT].apply(clean_base)

In [6]:
df_filtered[ProcessedData.TEXT_DL] = df_filtered[ProcessedData.FULL_TEXT].str.strip()

In [8]:
MAX_LEN = 200000  # o menys (ex: 100k)

def process_ml_safe(text: str) -> str:
    text = text[:MAX_LEN]  
    return process_ml(text)

df_filtered[ProcessedData.TEXT_ML] = df_filtered[ProcessedData.FULL_TEXT].apply(process_ml_safe)

## **3. Stratified Split and Export**

In [11]:
# Label Binarization
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df_filtered[ProcessedData.ODS_LIST])
ods_cols = mlb.classes_.tolist()

# Multi-label Stratified Split
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
train_val_idx, test_idx = next(msss.split(df_filtered[ProcessedData.FULL_TEXT].values, Y))

val_size_adj = VAL_SIZE / (1 - TEST_SIZE)
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_size_adj, random_state=RANDOM_SEED)
train_sub_idx, val_sub_idx = next(msss2.split(train_val_idx, Y[train_val_idx]))

train_idx, val_idx = train_val_idx[train_sub_idx], train_val_idx[val_sub_idx]

In [12]:
def export_data(dataframe, y_matrix, indices, folder, text_col):
    t_idx, v_idx, ts_idx = indices
    for name, idx in zip(['train', 'val', 'test'], [t_idx, v_idx, ts_idx]):
        subset = dataframe.iloc[idx][[text_col, Metadata.ODS_LIST]]
        labels_df = pd.DataFrame(y_matrix[idx], columns=ods_cols, index=subset.index)
        pd.concat([subset, labels_df], axis=1).to_parquet(os.path.join(folder, f'split_{name}.parquet'), index=False)

In [ ]:
# Export DL splits
export_data(df_filtered, Y, (train_idx, val_idx, test_idx), PROCESSED_DL_DIR, ProcessedData.TEXT_DL)

# Export ML splits + Vectorization
export_data(df_filtered, Y, (train_idx, val_idx, test_idx), PROCESSED_ML_DIR, ProcessedData.TEXT_ML)

tfidf = TfidfVectorizer(max_features=5000)
tfidf.fit(df_filtered.iloc[train_idx]['text_ml'])

with open(os.path.join(PROCESSED_ML_DIR, 'tfidf_model.pkl'), 'wb') as f: pickle.dump(tfidf, f)
with open(os.path.join(PROCESSED_ML_DIR, 'mlb.pkl'), 'wb') as f: pickle.dump(mlb, f)

print("Success: Data integrated, processed, and exported for both ML and DL.")

Success: Data integrated, processed, and exported for both ML and DL.
